In [ ]:
    ############    #############   Exception architecture   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Exception Architecture   #############   ##############   

 =>  Define domain-specific exceptions (NotFoundError, PermissionDeniedError) instead of
       raising bare Exception/ValueError everywhere -- callers can catch precisely what they
       expect.

 =>  Map each domain exception to an HTTP status once, in one place (an exception handler),
       instead of scattering try/except + status codes across every route.

 =>  Never leak internals (stack traces, SQL, file paths) in an API error response -- log the
       detail, return a safe, generic message to the client.


In [ ]:
class DomainError(Exception):
    """Base class for all expected, handled application errors."""

class NotFoundError(DomainError):
    def __init__(self, resource: str, resource_id: str):
        self.resource = resource
        self.resource_id = resource_id
        super().__init__(f"{resource} {resource_id} not found")

class PermissionDeniedError(DomainError):
    pass

EXCEPTION_STATUS_MAP = {
    NotFoundError: 404,
    PermissionDeniedError: 403,
    DomainError: 400,
}

def to_http_response(exc: DomainError) -> dict:
    for exc_type, status in EXCEPTION_STATUS_MAP.items():
        if isinstance(exc, exc_type):
            return {"status": status, "error": str(exc)}
    return {"status": 500, "error": "internal error"}

try:
    raise NotFoundError("document", "doc-123")
except DomainError as exc:
    print(to_http_response(exc))


In [ ]:
 =>  In FastAPI, 'to_http_response' becomes an '@app.exception_handler(DomainError)' --
       every route just raises the right domain exception and the mapping happens in
       exactly one place.

 =>  EXCEPTION_STATUS_MAP is checked in order -- put more specific exception types (like
       NotFoundError) before their more general parent (DomainError), or the specific case
       will never be reached.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Wire to_http_response's logic into a real FastAPI '@app.exception_handler(DomainError)'
           and confirm each domain exception returns the right status code end-to-end.

 =>  [ ] Add a ValidationError-style domain exception that returns 422 with a list of
           field-level problems, similar to how pydantic reports validation errors.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Catching bare 'except Exception' at the API boundary and returning the raw exception
       message to the client -- this can leak stack traces, file paths, or SQL fragments.

 =>  Defining domain exceptions but still using generic ValueError/RuntimeError in most of
       the actual code -- the hierarchy only helps if it's used consistently.
